In [ ]:
# CELL 1 — Imports & Session
# ===========================================================================
 
import json
import uuid
import time
from datetime import datetime
from snowflake.snowpark.context import get_active_session
 
session = get_active_session()
session.sql("USE WAREHOUSE EAGLE_WH").collect()
session.sql("USE DATABASE CITYLENS_MERGED_DB").collect()
session.sql("USE SCHEMA CITYLENS_SERVING").collect()
 
print(f"✅ Session ready!")
print(f"   Database  : {session.get_current_database()}")
print(f"   Schema    : {session.get_current_schema()}")
print(f"   Warehouse : {session.get_current_warehouse()}")

In [ ]:
# CELL 2 — Config (所有表名在这里，改名字只改这里)
# ===========================================================================
 
DB = "CITYLENS_MERGED_DB"
 
# Transportation SERVING tables
TRANSPORT_QA_CONTEXT       = f"{DB}.CITYLENS_SERVING.SRV_QA_CONTEXT"
TRANSPORT_WEATHER_CONTEXT  = f"{DB}.CITYLENS_SERVING.SRV_WEATHER_CONTEXT"
TRANSPORT_ANOMALY_CONTEXT  = f"{DB}.CITYLENS_SERVING.SRV_ANOMALY_CONTEXT"
TRANSPORT_BEST_TRAVEL_TIME = f"{DB}.CITYLENS_SERVING.SRV_BEST_TRAVEL_TIME"
TRANSPORT_RELIABILITY      = f"{DB}.CITYLENS_SERVING.SRV_ROUTE_RELIABILITY"
TRANSPORT_MONTHLY_TREND    = f"{DB}.CITYLENS_SERVING.SRV_MONTHLY_TREND"
TRANSPORT_STATION_RANKING  = f"{DB}.CITYLENS_SERVING.SRV_STATION_RISK_RANKING"
TRANSPORT_ROUTE_CONTEXT    = f"{DB}.CITYLENS_SERVING.SRV_ROUTE_CONTEXT"
 
# Transportation MART tables
TRANSPORT_ALERTS_SUMMARY   = f"{DB}.CITYLENS_MART.MART_ALERTS_SUMMARY"
TRANSPORT_ROUTE_WEEKLY     = f"{DB}.CITYLENS_MART.MART_ROUTE_WEEKLY"
TRANSPORT_STATION_HOURLY   = f"{DB}.CITYLENS_MART.MART_STATION_HOURLY"
TRANSPORT_TRAVEL_TIME      = f"{DB}.CITYLENS_MART.MART_TRAVEL_TIME_SUMMARY"
 
# Query log
TRANSPORT_QUERY_LOG = f"{DB}.CITYLENS_SERVING.QUERY_LOG"
 
print("✅ Config loaded!")
print(f"   DB: {DB}")
 
# 验证连接
result = session.sql(f"SELECT COUNT(*) AS CNT FROM {TRANSPORT_QA_CONTEXT}").collect()
print(f"   QA Context records: {result[0]['CNT']}")

In [ ]:
# CELL 3 — Router
# ===========================================================================
 
def route_query(user_question: str) -> dict:
    """
    识别线路、问题类型，决定调用哪些 analysts
    """
    question_upper = user_question.upper()
    question_lower = user_question.lower()
 
    # 识别线路
    lines = ["BLUE", "RED", "GREEN", "ORANGE", "SILVER"]
    found_line = ""
    for line in lines:
        if line in question_upper:
            found_line = line
            break
 
    # 识别问题类型
    if any(w in question_lower for w in ['best time', 'avoid crowd', 'quiet', 'when to ride']):
        intent = 'best_time'
    elif any(w in question_lower for w in ['reliable', 'reliability', 'on time', 'consistent']):
        intent = 'reliability'
    elif any(w in question_lower for w in ['trend', 'month', 'over time', 'trending']):
        intent = 'trend'
    elif any(w in question_lower for w in ['weather', 'rain', 'snow', 'storm']):
        intent = 'weather'
    elif any(w in question_lower for w in ['alert', 'delay', 'disruption', 'issue', 'problem']):
        intent = 'alerts'
    elif any(w in question_lower for w in ['station', 'stop', 'busy', 'crowded']):
        intent = 'station'
    elif any(w in question_lower for w in ['travel time', 'how long', 'duration', 'fast']):
        intent = 'travel_time'
    elif any(w in question_lower for w in ['anomaly', 'unusual', 'weird', 'spike', 'drop']):
        intent = 'anomaly'
    else:
        intent = 'general'
 
    # 决定调用哪些 analysts（所有问题都调用 performance + alerts）
    agents = ['performance_analyst', 'alerts_analyst']
 
    if intent in ['weather']:
        agents.append('weather_analyst')
    if intent in ['anomaly', 'alerts', 'general']:
        agents.append('anomaly_analyst')
    if intent in ['best_time', 'station']:
        agents.append('travel_time_analyst')
        agents.append('station_analyst')
    if intent in ['reliability', 'general']:
        agents.append('reliability_analyst')
    if intent in ['trend', 'general']:
        agents.append('monthly_analyst')
 
    routing_plan = {
        'query_id':       str(uuid.uuid4()),
        'user_query':     user_question,
        'intent':         intent,
        'found_line':     found_line,
        'agents_invoked': agents,
        'query_ts':       datetime.now().isoformat()
    }
 
    print(f"  Intent    : {intent}")
    print(f"  Line      : {found_line if found_line else 'All lines'}")
    print(f"  Agents    : {agents}")
    return routing_plan
 
print("✅ Router defined!")
 

In [ ]:
# CELL 4 — Analysts
# ===========================================================================
 
def _line_filter(found_line, prefix="WHERE"):
    """helper: 生成 line 过滤条件"""
    if found_line:
        return f"{prefix} UPPER(ROUTE_ID) LIKE '%{found_line}%'"
    return ""
 
def _entity_filter(found_line, prefix="WHERE"):
    if found_line:
        return f"{prefix} UPPER(ENTITY_NAME) LIKE '%{found_line}%'"
    return ""
 
 
def performance_analyst(routing_plan):
    found_line = routing_plan['found_line']
    query_text = routing_plan['user_query'].replace("'", "''")
    
    line_filter = f"AND UPPER(ENTITY_NAME) LIKE '%{found_line}%'" if found_line else ""
    
    results = session.sql(f"""
        SELECT 
            ENTITY_TYPE,
            ENTITY_NAME,
            TIME_PERIOD,
            SUMMARY_TEXT,
            VECTOR_COSINE_SIMILARITY(
                EMBEDDING,
                SNOWFLAKE.CORTEX.EMBED_TEXT_768(
                    'snowflake-arctic-embed-m', '{query_text}'
                )
            ) AS SIMILARITY
        FROM {TRANSPORT_QA_CONTEXT}
        WHERE EMBEDDING IS NOT NULL
        {line_filter}
        ORDER BY SIMILARITY DESC
        LIMIT 5
    """).collect()
    context_items = [
        {'entity': r['ENTITY_NAME'], 'summary': r['SUMMARY_TEXT'],
         'similarity': float(r['SIMILARITY'])}
        for r in results if r['SUMMARY_TEXT']
    ]
    return {'analyst': 'performance_analyst', 'retrieval_count': len(context_items), 'context': context_items} 
 
def weather_analyst(routing_plan):
    found_line = routing_plan['found_line']
    results = session.sql(f"""
        SELECT ROUTE_ID, SERVICE_DATE, WEATHER_CONDITION, AVG_TEMP_F,
               TOTAL_PRECIPITATION, TOTAL_EVENTS, SUMMARY_TEXT
        FROM {TRANSPORT_WEATHER_CONTEXT}
        {_line_filter(found_line)}
        ORDER BY SERVICE_DATE DESC
        LIMIT 5
    """).collect()
    context_items = [
        {'route': r['ROUTE_ID'], 'date': str(r['SERVICE_DATE']),
         'weather': r['WEATHER_CONDITION'], 'temp': r['AVG_TEMP_F'],
         'summary': r['SUMMARY_TEXT']}
        for r in results if r['SUMMARY_TEXT']
    ]
    return {'analyst': 'weather_analyst', 'retrieval_count': len(context_items), 'context': context_items}
 
 
def anomaly_analyst(routing_plan):
    found_line = routing_plan['found_line']
    results = session.sql(f"""
        SELECT ROUTE_ID, SERVICE_DATE, ANOMALY_TYPE, LIKELY_CAUSE,
               PCT_FROM_BASELINE, Z_SCORE, SUMMARY_TEXT
        FROM {TRANSPORT_ANOMALY_CONTEXT}
        {_line_filter(found_line)}
        ORDER BY ABS(Z_SCORE) DESC
        LIMIT 5
    """).collect()
    context_items = [
        {'route': r['ROUTE_ID'], 'date': str(r['SERVICE_DATE']),
         'anomaly_type': r['ANOMALY_TYPE'], 'likely_cause': r['LIKELY_CAUSE'],
         'z_score': r['Z_SCORE'], 'summary': r['SUMMARY_TEXT']}
        for r in results if r['SUMMARY_TEXT']
    ]
    return {'analyst': 'anomaly_analyst', 'retrieval_count': len(context_items), 'context': context_items}
 
 
def alerts_analyst(routing_plan):
    found_line = routing_plan['found_line']
    results = session.sql(f"""
        SELECT ROUTE_ID, CAUSE, EFFECT, SEVERITY,
               SUM(ALERT_COUNT) AS TOTAL_ALERTS
        FROM {TRANSPORT_ALERTS_SUMMARY}
        {_line_filter(found_line)}
        GROUP BY ROUTE_ID, CAUSE, EFFECT, SEVERITY
        ORDER BY TOTAL_ALERTS DESC
        LIMIT 5
    """).collect()
    context_items = [
        {'route': r['ROUTE_ID'], 'cause': r['CAUSE'],
         'effect': r['EFFECT'], 'severity': r['SEVERITY'],
         'total_alerts': int(r['TOTAL_ALERTS'])}
        for r in results if r['ROUTE_ID']
    ]
    return {'analyst': 'alerts_analyst', 'retrieval_count': len(context_items), 'context': context_items}
 
 
def travel_time_analyst(routing_plan):
    found_line = routing_plan['found_line']
    results = session.sql(f"""
        SELECT SUMMARY_TEXT
        FROM {TRANSPORT_BEST_TRAVEL_TIME}
        {_line_filter(found_line)}
        ORDER BY QUIETEST_RANK ASC
        LIMIT 5
    """).collect()
    context_items = [
        {'summary': r['SUMMARY_TEXT']}
        for r in results if r['SUMMARY_TEXT']
    ]
    return {'analyst': 'travel_time_analyst', 'retrieval_count': len(context_items), 'context': context_items}
 
 
def station_analyst(routing_plan):
    found_line = routing_plan['found_line']
    results = session.sql(f"""
        SELECT SUMMARY_TEXT
        FROM {TRANSPORT_STATION_RANKING}
        {_line_filter(found_line)}
        ORDER BY BUSIEST_RANK ASC
        LIMIT 5
    """).collect()
    context_items = [
        {'summary': r['SUMMARY_TEXT']}
        for r in results if r['SUMMARY_TEXT']
    ]
    return {'analyst': 'station_analyst', 'retrieval_count': len(context_items), 'context': context_items}
 
 
def reliability_analyst(routing_plan):
    found_line = routing_plan['found_line']
    results = session.sql(f"""
        SELECT ROUTE_ID, RELIABILITY_PCT, RELIABILITY_GRADE, SUMMARY_TEXT
        FROM {TRANSPORT_RELIABILITY}
        {_line_filter(found_line)}
        ORDER BY RELIABILITY_PCT DESC
        LIMIT 5
    """).collect()
    context_items = [
        {'route': r['ROUTE_ID'], 'reliability_pct': r['RELIABILITY_PCT'],
         'grade': r['RELIABILITY_GRADE'], 'summary': r['SUMMARY_TEXT']}
        for r in results if r['SUMMARY_TEXT']
    ]
    return {'analyst': 'reliability_analyst', 'retrieval_count': len(context_items), 'context': context_items}
 
 
def monthly_analyst(routing_plan):
    found_line = routing_plan['found_line']
    results = session.sql(f"""
        SELECT SUMMARY_TEXT
        FROM {TRANSPORT_MONTHLY_TREND}
        {_line_filter(found_line)}
        ORDER BY MONTH DESC
        LIMIT 4
    """).collect()
    context_items = [
        {'summary': r['SUMMARY_TEXT']}
        for r in results if r['SUMMARY_TEXT']
    ]
    return {'analyst': 'monthly_analyst', 'retrieval_count': len(context_items), 'context': context_items}
 
 
ANALYST_MAP = {
    'performance_analyst':  performance_analyst,
    'weather_analyst':      weather_analyst,
    'anomaly_analyst':      anomaly_analyst,
    'alerts_analyst':       alerts_analyst,
    'travel_time_analyst':  travel_time_analyst,
    'station_analyst':      station_analyst,
    'reliability_analyst':  reliability_analyst,
    'monthly_analyst':      monthly_analyst,
}
 
def run_analysts(routing_plan):
    results = {}
    total_retrievals = 0
    for agent_name in routing_plan['agents_invoked']:
        if agent_name in ANALYST_MAP:
            result = ANALYST_MAP[agent_name](routing_plan)
            results[agent_name] = result
            total_retrievals += result['retrieval_count']
            print(f"  ✅ {agent_name} — {result['retrieval_count']} items retrieved")
    return results, total_retrievals
 
print("✅ All analysts defined!")

In [ ]:
# CELL 5 — Answer Generator
# ===========================================================================

INTENT_INSTRUCTIONS = {
    'best_time':   "Identify the best times to ride based on crowding and trip data. Be specific about time of day and day of week.",
    'reliability': "Rank lines by reliability. Use reliability percentage and grade. Be specific with numbers.",
    'trend':       "Describe month-over-month trends. Note increases or decreases with specific percentages.",
    'weather':     "Explain how weather conditions affect performance. Use specific weather types and impact data.",
    'alerts':      "Summarize service alerts by cause and severity. Identify the most common issues.",
    'station':     "Identify the busiest stations and highest-risk time periods with specific event counts.",
    'travel_time': "Give specific travel times between key stops. Highlight worst and best performing segments.",
    'anomaly':     "Identify anomalies with Z-scores. Explain likely causes using the data.",
    'general':     "Give a comprehensive overview using all available data. Be specific with numbers.",
}

def generate_answer(routing_plan, analyst_results):

    # 统一处理所有不能序列化的类型
    def safe_serialize(obj):
        if hasattr(obj, 'isoformat'):  # date / datetime
            return obj.isoformat()
        return str(obj)

    context_text = ""
    for analyst_name, result in analyst_results.items():
        context_text += f"\n=== {analyst_name.upper()} ===\n"
        for item in result['context']:
            context_text += json.dumps(item, default=safe_serialize) + "\n"
    context_text = context_text[:5000]

    intent = routing_plan.get('intent', 'general')
    found_line = routing_plan.get('found_line', '')
    line_context = f"Focus on the {found_line} Line." if found_line else "Cover all MBTA lines."
    task_instruction = INTENT_INSTRUCTIONS.get(intent, INTENT_INSTRUCTIONS['general'])

    prompt = f"""You are a senior Boston MBTA transportation intelligence analyst.

Task: {task_instruction} {line_context}

Question: {routing_plan['user_query']}

Data (use ONLY the data provided below, do not make up routes, stations, or values):
{context_text}

IMPORTANT: Only use route names, station names, and values that appear in the data above.

Structure your answer as:
1. DIRECT ANSWER (1-2 sentences with specific numbers)
2. KEY FINDINGS (3-5 bullet points with data)
3. ROOT CAUSES (what likely caused any issues)
4. RECOMMENDATION (1-2 actionable suggestions)"""

    start_time = time.time()
    result = session.sql(f"""
        SELECT SNOWFLAKE.CORTEX.COMPLETE('claude-haiku-4-5', $${prompt}$$) AS ANSWER
    """).collect()
    latency_ms = int((time.time() - start_time) * 1000)

    return {
        'answer':     result[0]['ANSWER'],
        'tokens_in':  0,
        'tokens_out': 0,
        'latency_ms': latency_ms
    }

print("✅ Answer generator defined!")

In [ ]:
# CELL 6 — Reflection & Logger
# ===========================================================================
 
MBTA_LINES = ['blue', 'red', 'green', 'orange', 'silver']
MBTA_STATIONS = ['park street', 'downtown crossing', 'south station', 'north station',
                 'back bay', 'harvard', 'kendall', 'airport', 'alewife', 'forest hills']
 
def reflect_on_answer(answer, routing_plan):
    score = 0
 
    # 有具体数字
    if any(char.isdigit() for char in answer):
        score += 30
 
    # 提到了 MBTA 线路或站点
    answer_lower = answer.lower()
    if any(line in answer_lower for line in MBTA_LINES):
        score += 40
    elif any(station in answer_lower for station in MBTA_STATIONS):
        score += 40
 
    # 答案长度合理
    if 100 < len(answer) < 1500:
        score += 30
 
    return score
 
 
def log_query(routing_plan, answer_result, reflection_score, total_retrievals):
    try:
        clean_answer = answer_result['answer'][:500]
        clean_answer = clean_answer.replace("'", "''").replace("$", "\\$")
 
        session.sql(f"""
            INSERT INTO {TRANSPORT_QUERY_LOG} (
                QUERY_ID, USER_QUERY, QUERY_TS, INTENT,
                ENTITY_TYPE, ENTITY_NAME, TIME_WINDOW,
                AGENTS_INVOKED, TOKENS_IN, TOKENS_OUT,
                TOTAL_LATENCY_MS, GROUNDEDNESS_SCORE
            ) VALUES (
                '{routing_plan['query_id']}',
                '{routing_plan['user_query'].replace("'", "''")}',
                CURRENT_TIMESTAMP(),
                '{routing_plan['intent']}',
                'route',
                '{routing_plan['found_line'] or 'ALL'}',
                '2025',
                '{",".join(routing_plan['agents_invoked'])}',
                {answer_result['tokens_in']},
                {answer_result['tokens_out']},
                {answer_result['latency_ms']},
                {reflection_score}
            )
        """).collect()
        print(f"  ✅ Query logged: {routing_plan['query_id']}")
    except Exception as e:
        print(f"  ⚠️  Log failed: {e}")
 
print("✅ Reflection & Logger defined!")
 

In [ ]:
# CELL 7 — Main Agent Function
# ===========================================================================
 
def run_transport_agent(user_query: str) -> str:
    print(f"\n{'='*60}")
    print(f"❓ {user_query}")
    print('='*60)
 
    print("\n[Step 1] Routing...")
    routing_plan = route_query(user_query)
 
    print("\n[Step 2] Running analysts...")
    analyst_results, total_retrievals = run_analysts(routing_plan)
 
    print("\n[Step 3] Generating answer...")
    answer_result = generate_answer(routing_plan, analyst_results)
 
    print("\n[Step 4] Reflecting...")
    reflection_score = reflect_on_answer(answer_result['answer'], routing_plan)
    print(f"  Score: {reflection_score}/100")
 
    print("\n[Step 5] Logging...")
    log_query(routing_plan, answer_result, reflection_score, total_retrievals)
 
    print(f"\n{'='*60}")
    print("🤖 FINAL ANSWER:")
    print(answer_result['answer'])
    print(f"\n⏱  Latency    : {answer_result['latency_ms']}ms")
    print(f"📊 Retrievals : {total_retrievals}")
    print(f"⭐ Score      : {reflection_score}/100")
 
    return answer_result['answer']
 
print("✅ Transport agent ready!")
 

In [ ]:
# CELL 8 — Test
# ===========================================================================
 
test_questions = [
    "What is the best time to ride the Blue Line to avoid crowds?",
    "Which MBTA line is the most reliable?",
    "How has the Red Line been trending month over month?",
    "Why did the Orange Line have delays recently?",
    "Which stations are the busiest during morning rush hour?",
]
 
for q in test_questions:
    run_transport_agent(q)
    print()
 